# Part 2 — RAG narrative layer walkthrough

Shows the knowledge base, one narrated case (grounded, cited), and the
verification summary — by loading the artifacts `rag_narrate.py` produces and the
tested code in `src/oct_cds/rag/`. No reimplemented logic.

Generate the artifacts first (from the repo root):

```bash
pip install -e ".[rag]"
python rag_narrate.py paths=kaggle rag_run.split=external_test
```

Set `OCT_CDS_ENV` (`kaggle` or `default`). See [PART2.md](../PART2.md).

In [1]:
import json, os, sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

ENV = os.environ.get("OCT_CDS_ENV", "kaggle")
SPLIT = "external_test"

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

GlobalHydra.instance().clear()
with initialize_config_dir(version_base=None, config_dir=str(REPO / "configs")):
    cfg = OmegaConf.to_container(
        compose(config_name="config", overrides=[f"paths={ENV}", f"paths.root_dir={REPO.as_posix()}"]),
        resolve=True,
    )
GlobalHydra.instance().clear()

RAG_DIR = Path(cfg["output_dir"]) / "rag"
print("env      :", ENV)
print("rag dir  :", RAG_DIR)


def load_json(p):
    p = Path(p)
    if not p.exists():
        print(f"  (missing: {p})")
        return None
    return json.loads(p.read_text())


def show(df):
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df)

env      : kaggle
rag dir  : /kaggle/working/oct_cds_outputs/oct_c8_densenet121/rag


## 1 · Knowledge base

6 entries, NEI (public domain) + one CC-BY review. AAO Preferred Practice
Patterns and EyeWiki are deliberately excluded (their terms forbid use in an AI
system). Each `##` section is one retrievable, citable passage.

In [2]:
from oct_cds.rag.ingest import load_knowledge_base

kb = load_knowledge_base()
rows = []
for eid in sorted({p.entry_id for p in kb.passages}):
    ps = [p for p in kb.passages if p.entry_id == eid]
    rows.append({
        "entry": f"{eid}.md",
        "title": ps[0].entry_title,
        "covers": ", ".join(ps[0].covers),
        "passages": len(ps),
        "has_model_behavior_note": any(p.is_model_behavior_note for p in ps),
        "sources": " | ".join(ps[0].sources),
    })
show(pd.DataFrame(rows))
print(f"\n{len(kb.passages)} passages total; covers_map = {kb.covers_map}")
print("passage ids:", [p.id for p in kb.passages])

,entry,title,covers,passages,has_model_behavior_note,sources
0,amd.md,Age-Related Macular Degeneration and Drusen,"AMD, Drusen",6,True,National Eye Institute — Age-Related Macular D...
1,cnv.md,Choroidal Neovascularization (Wet AMD),CNV,6,True,National Eye Institute — Age-Related Macular D...
2,csr.md,Central Serous Retinopathy,CSR,6,True,"Varghese J, Kesharwani D, Parashar S, Agrawal ..."
3,dme.md,Diabetic Macular Edema,DME,6,True,"National Eye Institute — Macular Edema, nei.ni..."
4,dr.md,Diabetic Retinopathy,DR,6,True,"National Eye Institute — Diabetic Retinopathy,..."
5,macular_hole.md,Macular Hole,Macular Hole,6,True,"National Eye Institute — Macular Hole, nei.nih..."



36 passages total; covers_map = {'AMD': 'amd', 'Drusen': 'amd', 'CNV': 'cnv', 'CSR': 'csr', 'DME': 'dme', 'DR': 'dr', 'Macular Hole': 'macular_hole'}
passage ids: ['amd#overview', 'amd#symptoms', 'amd#risk-factors', 'amd#management', 'amd#referral', 'amd#model-behavior-note', 'cnv#overview', 'cnv#symptoms', 'cnv#risk-factors', 'cnv#management', 'cnv#referral', 'cnv#model-behavior-note', 'csr#overview', 'csr#symptoms', 'csr#risk-factors', 'csr#management', 'csr#referral', 'csr#model-behavior-note', 'dme#overview', 'dme#symptoms', 'dme#risk-factors', 'dme#management', 'dme#referral', 'dme#model-behavior-note', 'dr#overview', 'dr#symptoms', 'dr#risk-factors', 'dr#management', 'dr#referral', 'dr#model-behavior-note', 'macular_hole#overview', 'macular_hole#symptoms', 'macular_hole#risk-factors', 'macular_hole#management', 'macular_hole#referral', 'macular_hole#model-behavior-note']


## 2 · A narrated case

One case from `narratives_<split>.jsonl`. The **impression** and **triage** are
verbatim from Part 1's rule engine; `narrative_rag` is the model's grounded
explanation; every bracketed id resolves to a retrieved passage.

In [3]:
SAMPLE_STEM = None   # None -> first verified case; or set e.g. "CNV__p0"

jf = RAG_DIR / f"narratives_{SPLIT}.jsonl"
cases = [json.loads(l) for l in jf.read_text().splitlines() if l.strip()] if jf.exists() else []
if not cases:
    print(f"run:  python rag_narrate.py paths={ENV} rag_run.split={SPLIT}")
else:
    verified = [c for c in cases if c.get("narrator_meta", {}).get("verified")]
    pick = None
    if SAMPLE_STEM:
        pick = next((c for c in cases if Path(c["case"]["image_path"]).stem == SAMPLE_STEM), None)
    pick = pick or (verified[0] if verified else cases[0])

    m = pick["narrator_meta"]
    print("IMAGE     :", pick["case"]["image_path"], " true:", pick.get("true_class"))
    print("IMPRESSION: {predicted_class}  (confidence {confidence:.0%})  [from Part 1 rule engine]".format(**pick["impression"]))
    print("DIFFERENTIAL:", ", ".join(f"{d['class']} {d['probability']:.0%}" for d in pick["differential"]))
    print("TRIAGE    : {urgency}  — {recommendation}  [from Part 1 rule engine]".format(**pick["triage"]))
    print(f"VERIFIED  : {m['verified']}   flags: {m['flags']}   model: {m['model']}")
    print("\n── narrative_rag ──\n")
    print(pick.get("narrative_rag", "(fell back to Part 1 template)"))
    print("\n── citations ──")
    for c in pick.get("citations", []):
        print(f"  [{c['id']}]  {c['label']}")
        for s in c["sources"]:
            print(f"       - {s}")
    print("\nretrieved passage ids:", m["retrieved_ids"])

IMAGE     : /kaggle/input/datasets/pradamehavannan/bscans/bscans/CNV__p0.png  true: CNV
IMPRESSION: CNV  (confidence 98%)  [from Part 1 rule engine]
DIFFERENTIAL: CNV 98%, CSR 1%, Drusen 0%
TRIAGE    : urgent  — OCT features most consistent with CNV (calibrated probability 98%). Expedite specialist referral (24-72h).  [from Part 1 rule engine]
VERIFIED  : True   flags: []   model: Qwen/Qwen2.5-3B-Instruct

── narrative_rag ──

"The OCT scan shows choroidal neovascularization (CNV), a condition characterized by the growth of abnormal blood vessels beneath the retina, which can lead to rapid vision loss [cnv#overview]. This finding aligns with the model behavior note indicating strong performance on CNV classification with 93.4% sensitivity and 88.1% precision [cnv#model-behavior-note], though the external validation sample size was limited to just one case [cnv#model-behavior-note]. Given the high model confidence (98%) and the differential diagnosis suggesting CNV at 98%, central serou

## 3 · Verification summary

From `summary_<split>.json`. Every eligible case must produce a narrative that
passes `verify.py`, or it falls back to Part 1's template — nothing unverified is
ever shown.

In [4]:
s = load_json(RAG_DIR / f"summary_{SPLIT}.json")
if s:
    oc = s["outcomes"]
    show(pd.DataFrame(sorted(oc.items()), columns=["outcome", "n"]))
    eligible = sum(v for k, v in oc.items() if not k.startswith("skipped"))
    verified = oc.get("verified", 0) + oc.get("verified_with_flags", 0)
    fell_back = oc.get("fell_back", 0)
    print(f"\nbackend: {s['backend']}")
    print(f"eligible for narration: {eligible}")
    print(f"verified: {verified}/{eligible}"
          + (f"  ({verified/eligible:.0%})" if eligible else ""))
    print(f"fell back to Part 1 template: {fell_back}")
    print(f"flag counts: {s.get('flag_counts', {})}")
else:
    print(f"run:  python rag_narrate.py paths={ENV} rag_run.split={SPLIT}")

,outcome,n
0,skipped:Normal: no pathology nar,16
1,skipped:model abstained — no cla,6
2,verified,12
3,verified_with_flags,3



backend: hf_local
eligible for narration: 15
verified: 15/15  (100%)
fell back to Part 1 template: 0
flag counts: {'took 2 attempts': 3}


---

The language model explains; it never decides. The impression and triage in
every report come verbatim from Part 1's rule engine, and `verify.py` discards
any narrative that cites a passage it wasn't given, asserts a different class, or
softens the triage. Full write-up: [PART2.md](../PART2.md).